# Strict Split Model Comparison (Issue #36, Stage 1)

This notebook trains **all four models** on the frozen strict connected-component split
(`datasets/splits_strict/`, seed 32) and compares them on the strict validation split —
the same procedure as the original training notebooks
(`04_training/01_baseline_linear_regression.ipynb` … `04_catboost.ipynb`), with two
differences only:

1. the split is the strict connected-component split instead of the product-id split,
2. the per-model logic lives in `src/strict_funnel.py` instead of being copied into
   each notebook, so all four models fit in one notebook.

Each model runs its **known configurations** (the anchors carried over from the original
tuned workflow) across the trusted feature variants. **No random-search tuning happens
here.** Based on this comparison the two best models go to the tuning stage, and only
the final winner is evaluated once on the untouched test split
(`03_strict_final_holdout.ipynb`).

XGBoost and CatBoost early-stop on an inner component-grouped carve of the training
data — the validation split is never used for early stopping.

## 1. Imports

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "datasets").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.strict_funnel import (
    dummy_baselines,
    json_default,
    load_strict_training_frames,
    model_comparison_table,
    run_model_comparison,
)
from src.strict_protocol import COMPONENT_GROUP_COLUMN, build_identity_key

print("Repository root:", ROOT)

## 2. Run settings

Fixed before running. `QUICK` is only for smoke-testing the pipeline (tiny configs);
real runs leave `DPPM_QUICK` unset.

In [ ]:
QUICK = os.environ.get("DPPM_QUICK", "0") == "1"

ARTIFACT_DIR = ROOT / "artifacts" / ("strict_model_comparison_quick" if QUICK else "strict_model_comparison")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("Quick mode:", QUICK)
print("Artifacts:", ARTIFACT_DIR)

## 3. Load the frozen strict split

In [ ]:
train_df, validation_df = load_strict_training_frames(
    train_path=ROOT / "datasets/splits_strict/train_strict.csv",
    validation_path=ROOT / "datasets/splits_strict/validation_strict.csv",
)
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

## 4. Quick data checks

In [ ]:
print("Train price summary:")
print(train_df["price"].describe())
print()
print("Validation price summary:")
print(validation_df["price"].describe())

## 5. Split sanity checks

The frozen split already guarantees these; re-checked here so the notebook is
self-contained evidence.

In [ ]:
component_sizes = train_df[COMPONENT_GROUP_COLUMN].value_counts()
print("Components in train:", component_sizes.size)
print("Largest component rows:", int(component_sizes.max()))

train_keys = set(build_identity_key(train_df))
validation_keys = set(build_identity_key(validation_df))
overlap = train_keys & validation_keys
assert not overlap, f"Strict identity keys leak across train/validation: {len(overlap)}"
print("Identity-key overlap train/validation: 0 (PASS)")

## 6. Dummy anchors

Two trivial predictors scored on the strict validation split. Every model below must
clearly beat the subcategory-median guess to be worth reporting.

In [ ]:
best_summaries = {}
dummy_df = dummy_baselines(train_df, validation_df)
dummy_df.to_csv(ARTIFACT_DIR / "dummy_baselines.csv", index=False)
dummy_df

## 7. Ridge baseline (linear regression)

Log-price Ridge — the interpretable hedonic-style baseline. The original workflow tested OLS but selected Ridge with a small alpha, so the known configs are a small alpha grid around 0.05.

In [ ]:
best_summaries["ridge"] = run_model_comparison(
    "ridge",
    train_df,
    validation_df,
    output_dir=ARTIFACT_DIR / "ridge",
    quick=QUICK,
)
results_ridge = pd.read_csv(ARTIFACT_DIR / "ridge" / "comparison_results.csv")
results_ridge.head(10)

## 8. Random Forest

The five anchor configurations carried over from the original tuned Random Forest workflow.

In [ ]:
best_summaries["random_forest"] = run_model_comparison(
    "random_forest",
    train_df,
    validation_df,
    output_dir=ARTIFACT_DIR / "random_forest",
    quick=QUICK,
)
results_random_forest = pd.read_csv(ARTIFACT_DIR / "random_forest" / "comparison_results.csv")
results_random_forest.head(10)

## 9. XGBoost

The five anchor configurations from the original tuned XGBoost workflow.

In [ ]:
best_summaries["xgboost"] = run_model_comparison(
    "xgboost",
    train_df,
    validation_df,
    output_dir=ARTIFACT_DIR / "xgboost",
    quick=QUICK,
)
results_xgboost = pd.read_csv(ARTIFACT_DIR / "xgboost" / "comparison_results.csv")
results_xgboost.head(10)

## 10. CatBoost

The anchor configuration from the original CatBoost evaluation plus known small variants.

In [ ]:
best_summaries["catboost"] = run_model_comparison(
    "catboost",
    train_df,
    validation_df,
    output_dir=ARTIFACT_DIR / "catboost",
    quick=QUICK,
)
results_catboost = pd.read_csv(ARTIFACT_DIR / "catboost" / "comparison_results.csv")
results_catboost.head(10)

## 11. Model comparison

Best setup per model, ranked by validation MAE. This table decides which **two models**
go to the tuning stage.

In [ ]:
comparison_df = model_comparison_table(best_summaries)
comparison_df.to_csv(ARTIFACT_DIR / "model_comparison.csv", index=False)
(ARTIFACT_DIR / "best_summaries.json").write_text(
    json.dumps(best_summaries, indent=2, default=json_default) + "\n", encoding="utf-8"
)
print("Dummy anchors for context:")
print(dummy_df[["model_type", "validation_MAE", "validation_median_AE"]].to_string(index=False))
print()
comparison_df

## 12. What happens next

1. Pick the **two best models** from the table above and record the decision (with date)
   in `docs/DESIGN_DECISIONS.md`. *(Done 2026-07-07: Ridge and Random Forest advance;
   XGBoost and CatBoost eliminated - see the decision entries.)*
2. Tuning stage: full config search + component-grouped CV for the two finalists
   (notebook `02_strict_model_tuning.ipynb`, to be added next).
3. The tuning winner goes to `03_strict_final_holdout.ipynb` - run **once**.